***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 下一节： [9.2 基础校准流程](9_2_calibration_workflow.ipynb)

***


## 9.1 数据检查、可视化与初步质量控制

第 9 章把前面几章的理论对象串成一条实际处理链。连续谱数据处理通常从观测结构和数据质量检查开始，然后进入 flag、校准、成像、自校准和图像测量。这个顺序不是软件习惯的偶然结果，而是由误差传播决定的：若数据中已有明显坏天线、RFI 或错误扫描，后续求解器会把这些异常混入增益、带通或天空模型。

数据检查的目标不是寻找一张“好看的诊断图”，而是在开始求解之前回答三个问题：这份数据的时间、频率和基线覆盖是否满足科学目标；哪些异常属于必须 flag 的坏数据；哪些结构更可能是可校准的方向无关或方向依赖误差。只有先完成这种分类，后续校准才有物理意义。


### 9.1.1 从观测结构开始

真实 Measurement Set 中包含天线表、场表、谱窗、扫描、权重、flag 和可见度数据。即使使用 CASA 或其他软件，最先看的也不应该是成像结果，而是观测结构本身。天线布局决定基线长度分布，地球自转决定 $uv$ 轨迹，频率设置决定带宽和通道分辨率，扫描结构决定校准源能否跟踪目标场误差。

下面的合成数据保留了这些关键元素：六面天线、十五条基线、七十二个时间积分、六十四个频率通道、一个中心亮源和两个离轴弱源。数据中还故意注入了一段窄频 RFI 和一台天线的短时增益异常，用来展示早期 QA 应该识别的典型问题。


![合成测量集的天线布局与 uv 覆盖](figures/practical_ms_layout_uv.png)

**图 9.1.1** 合成测量集的天线布局与地球自转产生的 $uv$ 覆盖。阵列几何和观测时长决定了后续成像的分辨率、旁瓣结构和大尺度敏感性。


### 9.1.2 振幅随时间和频率的早期巡检

在数据质量检查中，振幅随时间、振幅随频率和动态谱是最有信息量的几类图。正常基线应呈现相对平滑的时间变化和带通结构；RFI 通常在某些时间或频率上产生尖锐异常；单天线链路问题则会同时影响所有含该天线的基线。把这些图按基线、天线和通道交叉比较，可以避免把一种误差误判成另一种误差。


![可见度振幅的时间、频率与动态谱检查](figures/practical_ms_visibility_qa.png)

**图 9.1.2** 正常基线和可疑基线的振幅诊断。短时窄频增强对应 RFI 候选；与 4 号天线相关的基线在后半段振幅系统下降，更像单天线链路异常。


### 9.1.3 初步 flag 统计

早期 flag 不应追求把所有细节一次处理干净，而应先隔离明显破坏统计分布的数据。一个朴素但有效的策略是：以每条基线的中位数和 MAD 估计正常振幅尺度，远高于正常水平的时频像素标记为 RFI 候选，长时间显著低于正常水平的基线段标记为链路异常候选。真实管线会使用更复杂的时频形态学和人工复核，但核心思想仍是先定位异常类型，再决定处理方式。


![基线和通道维度的 flag 占空比](figures/practical_ms_flag_summary.png)

**图 9.1.3** 初步 flag 规则得到的基线占空比和通道占空比。高占空比基线指向单天线问题，高占空比通道指向频率局部 RFI。


### 9.1.4 与真实软件流程的对应

这一节对应真实处理中的 `listobs`、`plotants`、`plotms`、`flagdata` 和 flag 复查。关键不是记住命令名，而是建立判断顺序：先确认观测结构，再识别明显异常，再决定哪些数据应当 flag，哪些误差应留给校准模型解释。并非所有异常都应立即删除；可校准的增益漂移、必须 flag 的坏数据和需要 3GC 的方向依赖残差，在处理链中的位置完全不同。
